# Практическая работа #2: Основы Python и анализ данных в Pandas
## Методический ориентир и эталонное решение (для преподавателя)

Данный блокнот содержит:
1. **Методическую структуру** лабораторной работы и демонстрацию сквозного пайплайна Data Cleaning & Analysis.
2. **Эталонное решение задач всех трех уровней сложности** (Easy, Medium, Hard) на реальных сгенерированных датасетах компьютерного клуба.
3. **Рекомендации по проверке**, разбор типичных студенческих ошибок и критерии оценивания.
4. **Задел на Задание №3** (структура будущих ненормализованных данных с JSON и вложенными строками).


## 0. Инициализация библиотек и настройка рабочего окружения


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Настройка читаемого отображения датафреймов
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print(f"Версия Pandas: {pd.__version__}")


Версия Pandas: 3.0.1


---
# 🟢 Часть 1. Разбор уровня сложности EASY (Базовый)
**Целевые навыки:** чтение CSV с разделителем `;`, выявление дубликатов, базовая очистка категориальных признаков через строковые методы и словари, заполнение пропусков, расчет производных колонок и простая группировка.


In [1]:
# 1.1. Загрузка данных
# Важно: акцентировать внимание студентов на параметре sep=';' и кодировке utf-8-sig
df_easy = pd.read_csv('easy/dataset_000.csv', sep=';')

print(f"Размерность исходного датасета: {df_easy.shape}")
print("\nПервые 3 строки:")
display(df_easy.head(3))

print("\nИнформация о типах данных и пропусках:")
df_easy.info()


NameError: name 'pd' is not defined

In [3]:
# 1.2. Поиск и удаление полных дубликатов
dup_count = df_easy.duplicated(subset=['session_id']).sum()
print(f"Найдено дубликатов по session_id: {dup_count}")

# Удаление дубликатов
df_easy = df_easy.drop_duplicates(subset=['session_id']).reset_index(drop=True)
print(f"Размерность после дедупликации: {df_easy.shape}")


Найдено дубликатов по session_id: 50
Размерность после дедупликации: (5000, 13)


In [4]:
# 1.3. Стандартизация категорий: pc_zone и gender
print("Уникальные значения pc_zone ДО очистки:", df_easy['pc_zone'].unique())

# Очистка pc_zone: удаление лишних пробелов и приведение регистра
df_easy['pc_zone'] = df_easy['pc_zone'].astype(str).str.strip()
df_easy['pc_zone'] = df_easy['pc_zone'].replace({
    'standard': 'Standard',
    'STANDARD': 'Standard',
    'VIP': 'VIP',
    'vip': 'VIP',
    'ps5': 'PS5',
    'PS5': 'PS5',
    'bootcamp': 'Bootcamp',
    'Bootcamp': 'Bootcamp'
})
print("Уникальные значения pc_zone ПОСЛЕ очистки:", df_easy['pc_zone'].unique())

print("\nУникальные значения gender ДО очистки:", df_easy['gender'].unique())
# Стандартизация пола через словарь замен (.map / .replace)
gender_map = {
    'M': 'M', 'm': 'M', '1': 'M', 'Мужской': 'M', ' M ': 'M', 'м': 'M',
    'F': 'F', 'f': 'F', '2': 'F', 'Женский': 'F', ' F ': 'F', 'ж': 'F'
}
df_easy['gender'] = df_easy['gender'].astype(str).str.strip().map(gender_map).fillna('M')
print("Уникальные значения gender ПОСЛЕ очистки:", df_easy['gender'].unique())


Уникальные значения pc_zone ДО очистки: <StringArray>
['VIP', 'Standard', 'PS5', 'Bootcamp', ' Bootcamp ', ' standard ', 'ps5', 'VIP ']
Length: 8, dtype: str
Уникальные значения pc_zone ПОСЛЕ очистки: <StringArray>
['VIP', 'Standard', 'PS5', 'Bootcamp']
Length: 4, dtype: str

Уникальные значения gender ДО очистки: <StringArray>
['M', 'F', ' M ', 'Мужской', '1', 'm', 'м', '2', 'ж', 'Женский']
Length: 10, dtype: str
Уникальные значения gender ПОСЛЕ очистки: <StringArray>
['M', 'F']
Length: 2, dtype: str


In [5]:
# 1.4. Обработка пропусков и анализ обратной связи
print("Количество пропусков по столбцам:")
print(df_easy.isna().sum()[df_easy.isna().sum() > 0])

# Заполнение технического пропуска в payment_method
df_easy["payment_method"] = df_easy["payment_method"].fillna("Unknown")

# Анализ поля рейтинга (необязательное поле обратной связи)
rated_share = df_easy["client_rating"].notna().mean() * 100
mean_rating = round(df_easy["client_rating"].mean(), 2)
print(f"\nДоля сессий с оценкой: {rated_share:.1f}%, пропусков: {100 - rated_share:.1f}%")
print(f"Средняя оценка среди оставивших отзыв: {mean_rating}")

# Заполнение пропусков рейтингом по умолчанию (средним) для дальнейших моделей
df_easy["client_rating"] = df_easy["client_rating"].fillna(mean_rating)
print(f"Осталось пропущенных значений в датасете: {df_easy.isna().sum().sum()}")


Количество пропусков по столбцам:
payment_method      33
client_rating     2771
dtype: int64

Доля сессий с оценкой: 44.6%, пропусков: 55.4%
Средняя оценка среди оставивших отзыв: 4.17
Осталось пропущенных значений в датасете: 0


In [16]:
# 1.5. Анализ данных Easy
# Создание расчетного столбца общей выручки
df_easy["total_paid"] = pd.to_numeric(df_easy["total_paid"])
df_easy["extra_bar_spent"] = pd.to_numeric(df_easy["extra_bar_spent"])
df_easy["total_revenue"] = df_easy["total_paid"] + df_easy["extra_bar_spent"]

print(f"Общая выручка клуба: {df_easy['total_revenue'].sum():,.0f} руб.")

# Комплексный анализ игровых зон: количество сессий, суммарная выручка и средний чек
zone_analysis = df_easy.groupby("pc_zone").agg(
    sessions_count=("session_id", "count"),
    total_revenue=("total_revenue", "sum"),
    avg_revenue_per_session=("total_revenue", "mean")
).sort_values("total_revenue", ascending=False)

# Добавление доли посещений для наглядности
zone_analysis["sessions_share_%"] = (
    zone_analysis["sessions_count"] / zone_analysis["sessions_count"].sum() * 100
).round(1)

print("\nСтатистика по игровым зонам (посещаемость, общая выручка и средний чек):")
display(zone_analysis)

# Топ-3 игр
print("\nТоп-3 самых популярных игр:")
print(df_easy["game_played"].value_counts().head(3))

# Распределение по полу
print("\nРаспределение клиентов по полу (%):")
print((df_easy["gender"].value_counts(normalize=True) * 100).round(1))

# Средний рейтинг по зонам
print("\nСредняя оценка клиентов по игровым зонам:")
print(df_easy.groupby("pc_zone")["client_rating"].mean().round(2))


Общая выручка клуба: 3,789,051 руб.

Статистика по игровым зонам (посещаемость, общая выручка и средний чек):
          sessions_count  total_revenue  avg_revenue_per_session  sessions_share_%
pc_zone                                                                           
Standard            2352        1285860                   546.71             47.00
VIP                 1015        1027739                  1012.55             20.30
Bootcamp            1113         869394                   781.13             22.30
PS5                  520         606058                  1165.50             10.40

Топ-3 самых популярных игр:
game_played
CS2         1595
Dota 2      1200
Valorant     652
Name: count, dtype: int64

Распределение клиентов по полу (%):
gender
M   82.00
F   18.00
Name: proportion, dtype: float64

Средняя оценка клиентов по игровым зонам:
pc_zone
Bootcamp   4.13
PS5        4.22
Standard   4.15
VIP        4.26
Name: client_rating, dtype: float64


---
# 🟡 Часть 2. Разбор уровня сложности MEDIUM (Средний)
**Целевые навыки:** очистка текстовых единиц из числовых полей регулярными выражениями, парсинг разнородных форматов дат через `pd.to_datetime(..., format='mixed')`, вычисление производного возраста на основе дат, булева фильтрация невозможных выбросов, дискретизация через `pd.cut`, построение сводных таблиц `pivot_table`.


In [18]:
# 2.1. Загрузка данных Medium
df_med = pd.read_csv('medium/dataset_000.csv', sep=';')
print(f"Размерность: {df_med.shape}")

# Базовая очистка дубликатов и пропусков
df_med = df_med.drop_duplicates(subset=['session_id']).reset_index(drop=True)
df_med['payment_method'] = df_med['payment_method'].fillna('Unknown')
df_med['client_rating'] = df_med['client_rating'].fillna(round(df_med['client_rating'].mean(), 2))
df_med['pc_zone'] = df_med['pc_zone'].astype(str).str.strip().replace({'ps5': 'PS5', 'standard': 'Standard', 'bootcamp': 'Bootcamp'})
df_med['gender'] = df_med['gender'].astype(str).str.strip().map(gender_map).fillna('M')


Размерность: (5100, 13)


In [19]:
# 2.2. Очистка строковых единиц в hours_played
print("Примеры значений hours_played ДО очистки:")
print(df_med['hours_played'].unique())

# Оставляем только цифры и знак минус (регулярное выражение)
df_med['hours_played'] = df_med['hours_played'].astype(str).str.replace(r'[^\d-]', '', regex=True)
# df_med['hours_played'] = df_med['hours_played'].astype(str).str.extract(r'(-?\d+)')
df_med['hours_played'] = pd.to_numeric(df_med['hours_played'])

print("\nУникальные значения hours_played ПОСЛЕ приведения к int:")
print(sorted(df_med['hours_played'].unique()))


Примеры значений hours_played ДО очистки:
<StringArray>
['1', '2', '3', '8', '12', '4', '~3', '5', '2 ч.', '999', '~12', '-2', '4 часа', '3 часа', '8 ч.', '1 часа', '8 часа', '~8', '4 ч.', '~1', '~2', '2 часа', '1 ч.', '3 ч.']
Length: 24, dtype: str

Уникальные значения hours_played ПОСЛЕ приведения к int:
[np.int64(-2), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(8), np.int64(12), np.int64(999)]


In [20]:
# 2.3. Парсинг дат и расчет возраста
# Приведение столбца date к datetime с поддержкой смешанных форматов
df_med['date'] = pd.to_datetime(df_med['date'], format='mixed')
df_med['birth_date'] = pd.to_datetime(df_med['birth_date'], format='mixed')

# Расчет возраста на 2026 год
df_med['age'] = 2026 - df_med['birth_date'].dt.year

print("Примеры вычисленного возраста:")
print(df_med[['birth_date', 'age']].head())


Примеры вычисленного возраста:
  birth_date  age
0 2004-04-19   22
1 2003-07-24   23
2 2004-06-28   22
3 2006-02-14   20
4 2009-02-09   17


In [21]:
# 2.4. Фильтрация аномалий (выбросов)
print(f"Строк до фильтрации: {len(df_med)}")

# Условие: сессии от 1 до 12 часов, возраст от 12 до 70 лет
valid_mask = (df_med['hours_played'].between(1, 12)) & (df_med['age'].between(12, 70))
df_med_clean = df_med[valid_mask].copy()

print(f"Строк после фильтрации аномалий: {len(df_med_clean)} (удалено {len(df_med) - len(df_med_clean)} строк)")


Строк до фильтрации: 5000
Строк после фильтрации аномалий: 4972 (удалено 28 строк)


In [26]:
# 2.5. Анализ Medium: сегментация и пиковые нагрузки
# Возрастные группы через pd.cut
df_med_clean["age_group"] = pd.cut(
    df_med_clean["age"],
    bins=[0, 17, 25, 100],
    labels=["до 18 лет", "18-25 лет", "старше 25 лет"]
)

# Расчет метрик по возрастным группам (бар, общий средний чек, суммарная выручка, топ-игра)
df_med_clean["total_paid"] = pd.to_numeric(df_med_clean["total_paid"])
df_med_clean["extra_bar_spent"] = pd.to_numeric(df_med_clean["extra_bar_spent"])
df_med_clean["total_revenue"] = df_med_clean["total_paid"] + df_med_clean["extra_bar_spent"]

age_summary = df_med_clean.groupby("age_group", observed=False).agg(
    avg_bar_spent=("extra_bar_spent", "mean"),
    avg_total_check=("total_revenue", "mean"),
    total_revenue=("total_revenue", "sum"),
    top_game=("game_played", lambda x: x.mode()[0])
)
print("Профиль возрастных групп (средний бар, общий средний чек, суммарная выручка и топ-игра):")
display(age_summary)

# Пиковые нагрузки: извлечение часа и дня недели
df_med_clean["hour"] = df_med_clean["date"].dt.hour
df_med_clean["day_name"] = df_med_clean["date"].dt.day_name()

# Календарный порядок дней недели (от Понедельника до Воскресенья)
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Сводная таблица загрузки (все 24 часа)
pivot_load = df_med_clean.pivot_table(
    index="day_name",
    columns="hour",
    values="session_id",
    aggfunc="count",
    fill_value=0
).reindex(days_order)

# Вывод полной таблицы на все 24 часа без усечения колонок
print("\nПолная сводная таблица загрузки клуба (сессий в час):")
with pd.option_context("display.max_columns", 25, "display.width", 1000):
    display(pivot_load)


Профиль возрастных групп (средний бар, общий средний чек, суммарная выручка и топ-игра):
               avg_bar_spent  avg_total_check  total_revenue top_game
age_group                                                            
до 18 лет              92.65           517.01         674183      CS2
18-25 лет             298.29           769.23        2073850      CS2
старше 25 лет         615.66          1193.94        1160511      CS2

Полная сводная таблица загрузки клуба (сессий в час):
hour       0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17   18   19  20  21  22  23
day_name                                                                                                   
Monday     17  10   7   6   0   2   3   8   6  16  18  23  48  23  37  27  46  53   57   37  40  40  17  25
Tuesday    21  10   6   3   0   1   4   7   9  15  18  20  23  19  37  37  45  41   60   41  29  31  28  16
Wednesday  15   5   7   9   2   3   2   5  10  16  16  26  30  28  32  39 

---
# 🔴 Часть 3. Разбор уровня сложности HARD (Продвинутый)
**Целевые навыки:** обнаружение и восстановление эффекта Row Shift (сдвига колонок), перерасчет утраченных бизнес-метрик по тарифам, распознавание и исправление кириллических подмен ('О' -> '0'), очистка первичных ключей, когортный анализ удержания (Retention) и LTV.


In [31]:
# 3.1. Загрузка данных Hard
df_hard = pd.read_csv('hard/dataset_000.csv', sep=';')
print(f"Исходная размерность: {df_hard.shape}")
print("Пропуски по столбцам:")
print(df_hard.isna().sum()[df_hard.isna().sum() > 0])


Исходная размерность: (5150, 13)
Пропуски по столбцам:
session_id          45
user_id             17
birth_date          46
pc_zone             46
payment_method      46
client_rating     2752
dtype: int64


In [32]:
# 3.2. Восстановление сдвинутых колонок (Row Shift)
# Диагностика: в pc_zone пропуск (NaN), а в hours_played находится название зоны!
known_zones = ["Standard", "VIP", "Bootcamp", "PS5"]
shift_mask = df_hard["pc_zone"].isna() & df_hard["hours_played"].isin(known_zones)
print(f"Обнаружено смещенных строк: {shift_mask.sum()}")

print("\nПример смещенной строки ДО исправления:")
display(df_hard[shift_mask][["session_id", "pc_zone", "hours_played", "extra_bar_spent", "total_paid", "discount_promo"]].head(2))

# Приведение к object для безопасного присвоения разнотипных значений при сдвиге (для совместимости с Pandas 3.0)
for col in ["pc_zone", "hours_played", "extra_bar_spent", "total_paid"]:
    df_hard[col] = df_hard[col].astype(object)

# 1) Название зоны возвращаем в pc_zone
df_hard.loc[shift_mask, "pc_zone"] = df_hard.loc[shift_mask, "hours_played"]
# 2) Часы игры возвращаем в hours_played
df_hard.loc[shift_mask, "hours_played"] = df_hard.loc[shift_mask, "extra_bar_spent"]
# 3) Траты бара возвращаем в extra_bar_spent
df_hard.loc[shift_mask, "extra_bar_spent"] = df_hard.loc[shift_mask, "total_paid"]

# 4) ВАЖНО: total_paid был затерт суммой бара, пересчитываем его по тарифу!
tariff_dict = {"Standard": 100, "Bootcamp": 150, "VIP": 200, "PS5": 250}
for idx in df_hard[shift_mask].index:
    zone = df_hard.loc[idx, "pc_zone"]
    hours = int(df_hard.loc[idx, "hours_played"])
    promo = int(df_hard.loc[idx, "discount_promo"])
    rate = tariff_dict.get(zone, 100)
    cost = hours * rate
    if promo == 1:
        cost = int(cost * 0.9)
    df_hard.loc[idx, "total_paid"] = cost

print("\nТа же строка ПОСЛЕ восстановления:")
display(df_hard[shift_mask][["session_id", "pc_zone", "hours_played", "extra_bar_spent", "total_paid", "discount_promo"]].head(2))


Обнаружено смещенных строк: 46

Пример смещенной строки ДО исправления:


,session_id,pc_zone,hours_played,extra_bar_spent,total_paid,discount_promo
43,SESS-13018,NaN,Bootcamp,4,400,0
79,SESS-13241,NaN,Standard,1,120,0



Та же строка ПОСЛЕ восстановления:


,session_id,pc_zone,hours_played,extra_bar_spent,total_paid,discount_promo
43,SESS-13018,Bootcamp,4,400,600,0
79,SESS-13241,Standard,1,120,100,0


In [33]:
# 3.3. Очистка кириллицы ('О') и мусора в total_paid
# Замена русской 'О' (Unicode 041E) на цифру '0' и удаление 'руб.', '₽'
df_hard["total_paid"] = df_hard["total_paid"].astype(str).str.replace("О", "0")
df_hard["total_paid"] = df_hard["total_paid"].str.replace(r"[^\d.]", "", regex=True)
df_hard["total_paid"] = pd.to_numeric(df_hard["total_paid"])

df_hard["extra_bar_spent"] = pd.to_numeric(df_hard["extra_bar_spent"])
df_hard["hours_played"] = pd.to_numeric(df_hard["hours_played"].astype(str).str.replace(r"[^\d-]", "", regex=True))

# Базовая стандартизация категорий и пропусков для Hard
df_hard["gender"] = df_hard["gender"].astype(str).str.strip().map(gender_map).fillna("M")
df_hard["pc_zone"] = df_hard["pc_zone"].astype(str).str.strip().replace({"ps5": "PS5", "standard": "Standard", "bootcamp": "Bootcamp"})
df_hard["payment_method"] = df_hard["payment_method"].fillna("Unknown")
df_hard["client_rating"] = df_hard["client_rating"].fillna(round(df_hard["client_rating"].mean(), 2))

# Удаление потерянных ключей и дубликатов
df_hard = df_hard.dropna(subset=["session_id", "user_id", "birth_date"]).reset_index(drop=True)
df_hard = df_hard.drop_duplicates(subset=["session_id"]).reset_index(drop=True)

# Фильтрация невозможных значений часов
df_hard = df_hard[df_hard["hours_played"].between(1, 12)].reset_index(drop=True)

# Приведение дат и расчет возраста
df_hard["date"] = pd.to_datetime(df_hard["date"], format="mixed")
df_hard["birth_date"] = pd.to_datetime(df_hard["birth_date"], format="mixed")
df_hard["age"] = 2026 - df_hard["birth_date"].dt.year
df_hard = df_hard[df_hard["age"].between(12, 70)].reset_index(drop=True)

# Итоговая выручка
df_hard["total_revenue"] = df_hard["total_paid"] + df_hard["extra_bar_spent"]

print(f"Размерность полностью чистого Hard датасета: {df_hard.shape}")


Размерность полностью чистого Hard датасета: (4865, 15)


In [35]:
# 3.4. Анализ Hard: Retention, LTV и влияние промокодов
# 1. Retention: визиты пользователей
visits = df_hard.groupby('user_id')['session_id'].count()
unique_users = len(visits)
single_visit_share = (visits == 1).mean() * 100
multi_visit_share = (visits > 1).mean() * 100

print(f"Уникальных пользователей: {unique_users}")
print(f"Доля клиентов с 1 посещением: {single_visit_share:.1f}%")
print(f"Доля клиентов с 2+ посещениями (постоянные): {multi_visit_share:.1f}%")

# 2. LTV (Топ-5 самых ценных клиентов)
top_clients = df_hard.groupby('user_id').agg(
    total_spent=('total_revenue', 'sum'),
    visits=('session_id', 'count'),
    avg_bill=('total_revenue', 'mean'),
    age=('age', 'first'),
    gender=('gender', 'first')
).sort_values('total_spent', ascending=False).head(5)

print("\nТоп-5 клиентов по LTV (суммарные траты за месяц):")
display(top_clients)

# 3. Проверка гипотезы о промокодах
promo_effect = df_hard.groupby('discount_promo').agg(
    sessions=('session_id', 'count'),
    mean_hours=('hours_played', 'mean'),
    mean_bar_spent=('extra_bar_spent', 'mean'),
    mean_total_revenue=('total_revenue', 'mean')
)
promo_effect.index = ['Без промокода (0)', 'С промокодом (1)']
print("\nВлияние промокода на поведение игроков:")
display(promo_effect)


Уникальных пользователей: 793
Доля клиентов с 1 посещением: 4.8%
Доля клиентов с 2+ посещениями (постоянные): 95.2%

Топ-5 клиентов по LTV (суммарные траты за месяц):


,total_spent,visits,avg_bill,age,gender
user_id,,,,,
USR-1088,30468.00,17,1792.24,28,M
USR-1010,28883.00,19,1520.16,27,M
USR-1136,24432.00,22,1110.55,31,M
USR-1083,22935.00,14,1638.21,34,M
USR-1012,20951.00,11,1904.64,37,M



Влияние промокода на поведение игроков:


,sessions,mean_hours,mean_bar_spent,mean_total_revenue
Без промокода (0),3795,3.10,226.47,687.79
С промокодом (1),1070,4.28,561.41,1128.11


---
# 🎓 Рекомендации для преподавателя по проверке работ

### Чек-лист критериев оценки:
1. **Качество кода:** использование векторизованных методов Pandas (`.str.replace()`, `.map()`, `.dropna()`, `.groupby()`) вместо ручных циклов `for i in range(...)`.
2. **Корректность обработки пропусков:** не удалены ли лишние строки при вызове `dropna()` без `subset` (это частая ошибка новичков, когда одна пропущенная оценка удаляет всю транзакцию).
3. **Обработка кириллической 'О':** обратил ли студент внимание на то, что `total_paid` не парсился в число из-за русской буквы.
4. **Восстановление сдвига строк:** сообразил ли студент, что значение `total_paid` нужно было пересчитать, а не просто забыть про него.
5. **Интерпретация результатов:** даны ли текстовые выводы к полученным таблицам и агрегациям.

---
# 🚀 Мостик к Заданию №3 (Вложенные структуры, JSON и распаковка)

Во 2-й работе студенты освоили базовый табличный Pandas. В Задании №3 та же модель компьютерного клуба расширится за счет ненормализованных полей:
* `bar_receipts` — строка формата JSON: `[{"item": "Monster", "qty": 2, "price": 180}, {"item": "Pringles", "qty": 1, "price": 220}]` -> задача распарсить через `json.loads` и распаковать через `.explode()`.
* `hardware_specs` — вложенный словарь характеристик ПК: `{"gpu": "RTX 4070", "cpu": "i7-13700", "monitor_hz": 240}` -> извлечение признаков.
* `games_log` — строковый лог активности: `"CS2:90min; Dota2:45min"` -> парсинг через `.str.split()` и регулярные выражения.
